# 🔁 TraceOps v0.6.0 — Interactive Demo

**TraceOps** records and replays LLM agent traces for deterministic, zero-cost regression testing.

This notebook walks through the entire library end-to-end:
1. Import & verify the library
2. Build traces manually
3. Record a mock agent run
4. Replay from a cassette (zero API calls)
5. Budget assertions
6. Diff two traces
7. Cost dashboard
8. HTML report generation
9. CLI commands
10. Performance benchmark
11. Edge cases & robustness
12. RAG recording & assertions
13. Semantic regression detection
14. MCP tool recording
15. Fine-tune dataset export
16. Behavioral pattern analysis
17. **🆕 LLM-as-Judge evaluation**
18. **🆕 Custom evaluation criteria**
19. **🆕 Eval CLI command**
20. Brainstorm: where to extend next

> **No real API keys required.** All LLM responses are mocked.

In [ ]:
# Install traceops from PyPI
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "traceops[openai]", "--quiet"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ traceops installed from PyPI")
else:
    print("❌ Install failed:")
    print(result.stderr[-2000:])


## 1. Import Required Libraries

In [ ]:
import sys, json, time, pathlib, subprocess, shutil
from pathlib import Path
from unittest.mock import MagicMock, patch

import trace_ops
from trace_ops import (
    Recorder, Replayer,
    Trace, TraceEvent, TraceMetadata, EventType,
    save_cassette, load_cassette,
    diff_traces, assert_trace_unchanged, TraceDiff,
    assert_cost_under, assert_tokens_under, assert_max_llm_calls, assert_no_loops,
    BudgetExceededError, AgentLoopError,
    CostDashboard,
)
from trace_ops.reporters.html import generate_html_report

print(f"✅ TraceOps {trace_ops.__version__} loaded")
print(f"   Python {sys.version.split()[0]}")


## 2. Build a Trace Manually

A `Trace` is a sequence of `TraceEvent` objects. You can build one directly — useful for understanding the data model or for testing your own agent framework integrations.

In [ ]:
# --- Build a realistic trace step-by-step ---

trace = Trace(metadata=TraceMetadata(description="Demo math agent", tags=["demo"]))

# Step 1 — user sends a message to the LLM
trace.add_event(TraceEvent(
    event_type=EventType.LLM_REQUEST,
    provider="openai",
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is the square root of 144? Use the calculator tool."},
    ],
    tools=[{"type": "function", "function": {"name": "calculator", "description": "Evaluates math expressions"}}],
    temperature=0.0,
))

# Step 2 — LLM responds with a tool call
trace.add_event(TraceEvent(
    event_type=EventType.LLM_RESPONSE,
    provider="openai",
    model="gpt-4o",
    response={"choices": [{"message": {"tool_calls": [{"function": {"name": "calculator", "arguments": '{"expr":"sqrt(144)"}'}}]}}]},
    input_tokens=55,
    output_tokens=20,
    cost_usd=0.00045,
    duration_ms=430.2,
))

# Step 3 — agent calls the calculator tool
trace.add_event(TraceEvent(
    event_type=EventType.TOOL_CALL,
    tool_name="calculator",
    tool_input={"expr": "sqrt(144)"},
))

# Step 4 — tool returns a result
trace.add_event(TraceEvent(
    event_type=EventType.TOOL_RESULT,
    tool_name="calculator",
    tool_output="12",
    duration_ms=1.3,
))

# Step 5 — LLM gives final answer
trace.add_event(TraceEvent(
    event_type=EventType.LLM_REQUEST,
    provider="openai",
    model="gpt-4o",
    messages=[{"role": "assistant", "content": "The answer is 12."}],
))
trace.add_event(TraceEvent(
    event_type=EventType.LLM_RESPONSE,
    provider="openai",
    model="gpt-4o",
    response={"choices": [{"message": {"content": "The square root of 144 is **12**."}}]},
    input_tokens=70,
    output_tokens=15,
    cost_usd=0.00030,
    duration_ms=310.7,
))

trace.finalize()

print(f"Trace ID   : {trace.trace_id}")
print(f"Events     : {len(trace.events)}")
print(f"LLM calls  : {trace.total_llm_calls}")
print(f"Tool calls : {trace.total_tool_calls}")
print(f"Tokens     : {trace.total_tokens:,}")
print(f"Cost       : ${trace.total_cost_usd:.5f}")
print(f"Duration   : {trace.total_duration_ms:.1f}ms")
print(f"Fingerprint: {trace.fingerprint()}")
print(f"\nTrajectory : {' → '.join(trace.trajectory)}")

## 3. Save & Load a Cassette

Cassettes are human-readable YAML files that can be committed to version control. API keys and secrets are automatically redacted.

In [ ]:
CASSETTE_DIR = Path("demo_cassettes")
CASSETTE_DIR.mkdir(exist_ok=True)

cassette_path = CASSETTE_DIR / "math_agent.yaml"

# Save
save_cassette(trace, cassette_path)
print(f"✅ Saved → {cassette_path}  ({cassette_path.stat().st_size / 1024:.1f} KB)")

# Peek at the raw YAML (first 40 lines)
raw = cassette_path.read_text(encoding="utf-8")
lines = raw.splitlines()
print(f"\n--- {cassette_path.name} (first 40 lines) ---")
print("\n".join(lines[:40]))
print("...")

In [ ]:
# Load it back
loaded = load_cassette(cassette_path)

print(f"Reloaded trace_id : {loaded.trace_id}")
print(f"Events deserialized: {len(loaded.events)}")
print(f"Fingerprint matches: {loaded.fingerprint() == trace.fingerprint()}")

# All event types round-trip correctly
for ev in loaded.events:
    print(f"  {ev.event_type.value:18s}  model={ev.model or '—':10s}  tool={ev.tool_name or '—'}")

## 4. Record a Mock Agent Run with `Recorder`

`Recorder` monkey-patches OpenAI / Anthropic / LiteLLM SDK methods at runtime. Here we mock the OpenAI client so no real API key is needed.

In [ ]:
import types, importlib

# ── Build a minimal fake openai module so Recorder can patch it ─────
fake_openai = types.ModuleType("openai")

class _FakeCompletion:
    def create(self, **kwargs):
        # Simulate a real OpenAI response object
        resp = MagicMock()
        resp.model = kwargs.get("model", "gpt-4o")
        resp.usage.prompt_tokens = 60
        resp.usage.completion_tokens = 25
        resp.usage.total_tokens = 85
        choice = MagicMock()
        choice.message.content = "The capital of France is **Paris**."
        choice.message.tool_calls = None
        choice.finish_reason = "stop"
        resp.choices = [choice]
        resp.id = "chatcmpl-demo-001"
        return resp

    async def acreate(self, **kwargs):
        return self.create(**kwargs)

class _FakeChat:
    completions = _FakeCompletion()

fake_openai.chat = _FakeChat()
fake_openai.OpenAI = MagicMock(return_value=MagicMock(chat=_FakeChat()))
sys.modules["openai"] = fake_openai

print("✅ Fake openai module injected")

In [ ]:
recorded_path = CASSETTE_DIR / "capital_agent.yaml"

with Recorder(
    description="Ask the capital of France",
    tags=["geo", "demo"],
    save_to=str(recorded_path),
    intercept_anthropic=False,
    intercept_litellm=False,
    intercept_langchain=False,
    intercept_langgraph=False,
    intercept_crewai=False,
) as rec:
    # Simulate an agent calling openai.chat.completions.create
    response = fake_openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "What is the capital of France?"}],
    )
    answer = response.choices[0].message.content

recorded_trace = rec.trace
print(f"🎙️  Recorded!")
print(f"   Answer captured : {answer}")
print(f"   Events recorded : {len(recorded_trace.events)}")
print(f"   LLM calls       : {recorded_trace.total_llm_calls}")
print(f"   Tokens          : {recorded_trace.total_tokens}")
print(f"   Cassette saved  : {recorded_path}")

## 5. Replay from a Cassette — Zero API Calls

`Replayer` patches the same SDK methods but returns stored responses instead of hitting the network. The agent code is completely unchanged.

In [ ]:
t0 = time.perf_counter()

with Replayer(
    str(recorded_path),
    strict=True,
    intercept_anthropic=False,
    intercept_litellm=False,
    intercept_langchain=False,
    intercept_langgraph=False,
    intercept_crewai=False,
) as rep:
    # Exact same agent code — no real API call happens
    response = fake_openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "What is the capital of France?"}],
    )
    replayed_answer = response.choices[0].message.content

elapsed = (time.perf_counter() - t0) * 1000

print(f"⚡ Replayed in {elapsed:.1f}ms  (vs ~400ms+ for real API)")
print(f"   Answer: {replayed_answer}")
print(f"   Matches original: {replayed_answer == answer}")

## 6. Budget Assertions

Enforce cost, token, and behavioural budgets on any `Trace`. These are the assertions you'd put in your CI pipeline.

In [ ]:
# ── Use the manually-built math trace from section 2 ────────────────

print("=== Assertions on 'math_agent' trace ===\n")
print(f"  Cost      : ${trace.total_cost_usd:.5f}")
print(f"  Tokens    : {trace.total_tokens}")
print(f"  LLM calls : {trace.total_llm_calls}")

# ✅ These should all pass
assert_cost_under(trace, max_usd=1.00)
print("✅ assert_cost_under($1.00)  — passed")

assert_tokens_under(trace, max_tokens=500)
print("✅ assert_tokens_under(500)  — passed")

assert_max_llm_calls(trace, max_calls=5)
print("✅ assert_max_llm_calls(5)   — passed")

assert_no_loops(trace, max_consecutive_same_tool=3)
print("✅ assert_no_loops(3)        — passed")

# ❌ Now trigger a BudgetExceededError deliberately
print("\n=== Triggering BudgetExceededError ===\n")
try:
    assert_cost_under(trace, max_usd=0.0001)   # very tight budget
except BudgetExceededError as e:
    print(f"💥 BudgetExceededError caught:\n   {str(e).splitlines()[0]}")

# ❌ Build a loop trace and catch AgentLoopError
loopy = Trace()
for _ in range(5):
    loopy.add_event(TraceEvent(event_type=EventType.TOOL_CALL, tool_name="search"))
loopy.finalize()

try:
    assert_no_loops(loopy, max_consecutive_same_tool=3)
except AgentLoopError as e:
    print(f"\n💥 AgentLoopError caught:\n   {str(e).splitlines()[0]}")

## 7. Diff Two Traces

`diff_traces` compares two traces and returns a structured `TraceDiff` — trajectory changes, model swaps, token deltas, response content diffs.

In [ ]:
# --- Build a "new version" trace where the agent switched models ----

trace_v2 = Trace(metadata=TraceMetadata(description="Math agent v2 — switched to gpt-4o-mini"))

trace_v2.add_event(TraceEvent(
    event_type=EventType.LLM_REQUEST, provider="openai", model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is sqrt(144)?"}],
))
trace_v2.add_event(TraceEvent(
    event_type=EventType.LLM_RESPONSE, provider="openai", model="gpt-4o-mini",
    response={"choices": [{"message": {"tool_calls": [{"function": {"name": "calculator", "arguments": '{"expr":"sqrt(144)"}'}}]}}]},
    input_tokens=40, output_tokens=18, cost_usd=0.00010, duration_ms=180.0,
))
trace_v2.add_event(TraceEvent(event_type=EventType.TOOL_CALL, tool_name="calculator", tool_input={"expr": "sqrt(144)"}))
trace_v2.add_event(TraceEvent(event_type=EventType.TOOL_RESULT, tool_name="calculator", tool_output="12"))
# v2 skips the second LLM call — fewer round-trips!
trace_v2.finalize()

# --- Diff ---
result = diff_traces(trace, trace_v2)

print(f"has_changes       : {result.has_changes}")
print(f"trajectory_changed: {result.trajectory_changed}")
print(f"llm_calls_delta   : {result.llm_calls_delta:+d}  ({trace.total_llm_calls} → {trace_v2.total_llm_calls})")
print(f"token_delta       : {result.token_delta:+d}    ({trace.total_tokens} → {trace_v2.total_tokens})")
print(f"cost_delta        : ${result.cost_delta:+.5f}")
print(f"changed_models    : {result.changed_models}")
print()
print("─── Summary ────────────────────────────────")
print(result.summary())

In [ ]:
# Save both as cassettes so we can diff via CLI later
save_cassette(trace,    CASSETTE_DIR / "math_v1.yaml")
save_cassette(trace_v2, CASSETTE_DIR / "math_v2.yaml")

# assert_trace_unchanged — this should raise because the trajectory changed
print("Calling assert_trace_unchanged (should raise)...")
try:
    assert_trace_unchanged(trace, trace_v2, ignore_trajectory=False, ignore_costs=True)
except AssertionError as e:
    print(f"✅ AssertionError raised as expected:\n   {str(e).splitlines()[0]}")

## 8. Cost Dashboard

`CostDashboard` scans a directory of cassettes and produces aggregate spend metrics — total cost, per-model breakdown, top cassettes by cost.

In [ ]:
dashboard = CostDashboard(CASSETTE_DIR)
summary = dashboard.data

print(f"Cassettes scanned : {summary.cassette_count}")
print(f"Total LLM calls   : {summary.total_llm_calls}")
print(f"Total tokens      : {summary.total_tokens:,}")
print(f"Total cost        : ${summary.total_cost_usd:.5f}")

print("\n── Cost by Model ──────────────────────────────")
for m in summary.by_model:
    print(f"  {m.model:20s}  calls={m.calls}  tokens={m.total_tokens:,}  cost=${m.cost_usd:.5f}")

print("\n── Top Cassettes by Cost ──────────────────────")
for c in summary.by_cassette:
    print(f"  {c.path:25s}  llm={c.llm_calls}  tokens={c.total_tokens:,}  cost=${c.cost_usd:.5f}")

# Also available as plain dict (e.g. for JSON logging in CI)
d = summary.to_dict()
print(f"\nJSON keys: {list(d.keys())}")

## 9. Generate an HTML Report

`generate_html_report` produces a self-contained, dark-themed HTML file with an interactive event timeline — no external dependencies, just open in a browser.

In [ ]:
report_path = CASSETTE_DIR / "math_report.html"
generate_html_report(trace, report_path, compare_trace=trace_v2, title="Math Agent — v1 vs v2")

size_kb = report_path.stat().st_size / 1024
print(f"✅ HTML report written: {report_path}  ({size_kb:.1f} KB)")
print(f"   Open in browser to view the interactive timeline + diff")

# Quick sanity check on the generated HTML
html = report_path.read_text(encoding="utf-8")
for marker in ["Trajectory", "Token", "Cost", "Diff", "gpt-4o"]:
    status = "✅" if marker in html else "❌"
    print(f"   {status}  section '{marker}' present")

## 10. CLI Commands

Every feature is also available from the terminal via the `traceops` command. Let's call a few inline to see the output.


In [ ]:
def run_cli(*args):
    """Run a traceops CLI command and print the output."""
    cmd = [sys.executable, "-m", "trace_ops.cli"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    print(output or "(no output)")
    return result

print("=" * 60)
print("traceops inspect demo_cassettes/math_agent.yaml")
print("=" * 60)
run_cli("inspect", "demo_cassettes/math_agent.yaml")


In [ ]:
print("=" * 60)
print("replay diff demo_cassettes/math_v1.yaml demo_cassettes/math_v2.yaml --detailed")
print("=" * 60)
run_cli("diff", "demo_cassettes/math_v1.yaml", "demo_cassettes/math_v2.yaml", "--detailed")

In [ ]:
print("=" * 60)
print("replay costs demo_cassettes/ --json")
print("=" * 60)
run_cli("costs", "demo_cassettes/", "--json")

In [ ]:
print("=" * 60)
print("replay validate demo_cassettes/math_agent.yaml")
print("=" * 60)
run_cli("validate", "demo_cassettes/math_agent.yaml")

print()
print("=" * 60)
print("replay stats demo_cassettes/")
print("=" * 60)
run_cli("stats", "demo_cassettes/")

## 11. Performance Benchmark

Compare replay speed vs. a simulated real API call latency.

In [ ]:
N = 20  # number of iterations

# Simulate a "real" API call with network latency
SIMULATED_LATENCY_MS = 350

real_times = []
for _ in range(N):
    t0 = time.perf_counter()
    time.sleep(SIMULATED_LATENCY_MS / 1000)   # pretend network round-trip
    real_times.append((time.perf_counter() - t0) * 1000)

# Measure actual replay times
replay_times = []
for _ in range(N):
    t0 = time.perf_counter()
    loaded = load_cassette(cassette_path)  # load + deserialize
    replay_times.append((time.perf_counter() - t0) * 1000)

avg_real   = sum(real_times)   / N
avg_replay = sum(replay_times) / N
speedup    = avg_real / avg_replay

print(f"{'Metric':<30} {'Real API (simulated)':>22}  {'Replay':>10}  {'Speedup':>10}")
print("─" * 75)
print(f"{'Average latency (ms)':<30} {avg_real:>22.1f}  {avg_replay:>10.2f}  {speedup:>9.0f}x")
print(f"{'Total for {N} calls (ms)':<30} {sum(real_times):>22.0f}  {sum(replay_times):>10.1f}")
print(f"\n🚀 Replay is ~{speedup:.0f}x faster  |  saves ~${trace.total_cost_usd * N:.4f} per {N} test runs")

## 12. Edge Cases & Robustness

Quickly exercise a few edge cases: empty trace, error events, missing cassette, non-strict replay.

In [ ]:
from trace_ops.cassette import CassetteNotFoundError

# ── Empty trace ─────────────────────────────────────────
empty = Trace()
empty.finalize()
print(f"Empty trace   — events={len(empty.events)}, cost=${empty.total_cost_usd}")
print(f"  trajectory  : {empty.trajectory}  (empty list)")
print(f"  fingerprint : {empty.fingerprint()}")

# ── Error event ─────────────────────────────────────────
err_trace = Trace()
err_trace.add_event(TraceEvent(
    event_type=EventType.ERROR,
    error_type="RateLimitError",
    error_message="You exceeded your current quota",
    metadata={"retry_after": 60},
))
err_trace.finalize()
print(f"\nError trace   — events={len(err_trace.events)}")
print(f"  error_type  : {err_trace.events[0].error_type}")

# ── Missing cassette ────────────────────────────────────
try:
    load_cassette("cassettes/does_not_exist.yaml")
except CassetteNotFoundError as e:
    print(f"\n✅ CassetteNotFoundError: {str(e).splitlines()[0]}")

# ── Diff of identical traces ────────────────────────────
same_diff = diff_traces(trace, trace)
print(f"\nDiff with itself — has_changes={same_diff.has_changes}")
print(f"  {same_diff.summary()}")


## 12. 🆕 RAG Recording & Assertions

TraceOps v0.5.0 can record retrieval-augmented generation (RAG) events — what chunks were fetched, their scores, and whether the context was actually used in the response.

In [ ]:
from trace_ops.rag import RAGRecorder
from trace_ops.rag.assertions import (
    assert_min_chunks_retrieved,
    assert_max_chunks_retrieved,
    assert_min_relevance_score,
    assert_chunk_used_in_response,
)

# Build a RAG trace manually (simulating a retriever returning chunks)
rag_recorder = RAGRecorder()

with rag_recorder.record_retrieval(
    query="What is the capital of France?",
    retriever="demo_retriever",
) as retrieval:
    # Simulate chunks returned by a vector store
    retrieval.add_chunk(
        content="Paris is the capital and most populous city of France.",
        score=0.97,
        metadata={"source": "wiki/france.txt", "chunk_id": "c001"},
    )
    retrieval.add_chunk(
        content="France is a country in Western Europe.",
        score=0.82,
        metadata={"source": "wiki/france.txt", "chunk_id": "c002"},
    )
    retrieval.add_chunk(
        content="The Eiffel Tower is located in Paris.",
        score=0.74,
        metadata={"source": "wiki/eiffel.txt", "chunk_id": "c003"},
    )

rag_trace = rag_recorder.trace
print(f"RAG trace events : {len(rag_trace.events)}")
retrieval_event = rag_trace.events[0]
print(f"Query            : {retrieval_event.data['query']}")
print(f"Chunks retrieved : {len(retrieval_event.data['chunks'])}")
for i, chunk in enumerate(retrieval_event.data['chunks']):
    print(f"  [{i+1}] score={chunk['score']:.2f}  {chunk['content'][:55]}...")

# ── Assertions ───────────────────────────────────────────
print("\n=== RAG Assertions ===")
assert_min_chunks_retrieved(rag_trace, min_chunks=2)
print("✅ assert_min_chunks_retrieved(2)  — passed")

assert_max_chunks_retrieved(rag_trace, max_chunks=10)
print("✅ assert_max_chunks_retrieved(10) — passed")

assert_min_relevance_score(rag_trace, min_score=0.70)
print("✅ assert_min_relevance_score(0.70) — passed")

assert_chunk_used_in_response(
    rag_trace,
    response_text="The capital of France is Paris.",
    min_overlap_words=1,
)
print("✅ assert_chunk_used_in_response   — passed")

# ❌ Trigger a failure
try:
    assert_min_relevance_score(rag_trace, min_score=0.99)
except AssertionError as e:
    print(f"\n💥 AssertionError (low relevance): {str(e).splitlines()[0]}")

## 13. 🆕 Semantic Regression Detection

Instead of checking exact string matches, `semantic_similarity` computes cosine similarity between two texts. Use it to catch regressions where the *meaning* of a response changed even if the words differ.

In [ ]:
from trace_ops.semantic.similarity import semantic_similarity, cosine_similarity
from trace_ops.semantic.assertions import assert_semantically_similar, assert_semantically_different

# --- Semantic similarity between two strings (no OpenAI key needed for TF-IDF fallback) ---
pairs = [
    ("The capital of France is Paris.",        "Paris is the capital city of France."),
    ("The capital of France is Paris.",        "London is the capital of the United Kingdom."),
    ("The weather today is sunny and warm.",   "It is a bright and warm day outside."),
    ("The weather today is sunny and warm.",   "The stock market fell 3% on Monday."),
]

print(f"{'Text A'[:45]:<45}  {'Text B'[:45]:<45}  Similarity")
print("─" * 110)
for a, b in pairs:
    score = cosine_similarity(a, b)
    bar = "█" * int(score * 20)
    print(f"{a[:45]:<45}  {b[:45]:<45}  {score:.3f} {bar}")

# --- Semantic diff on full traces ---
print("\n=== Semantic Diff of Traces ===")
result_semantic = diff_traces(trace, trace_v2, semantic=True)
print(f"has_changes        : {result_semantic.has_changes}")
print(f"semantic_regressions: {result_semantic.semantic_regressions}")
print(f"summary: {result_semantic.summary()}")

# --- Assertions ---
print("\n=== Semantic Assertions ===")
assert_semantically_similar(
    "The capital of France is Paris.",
    "Paris is the capital city of France.",
    threshold=0.70,
)
print("✅ assert_semantically_similar — passed")

assert_semantically_different(
    "The capital of France is Paris.",
    "The stock market fell 3% on Monday.",
    threshold=0.50,
)
print("✅ assert_semantically_different — passed")

try:
    assert_semantically_similar(
        "The capital of France is Paris.",
        "The stock market fell 3% on Monday.",
        threshold=0.90,
    )
except AssertionError as e:
    print(f"\n💥 AssertionError (not similar enough): {str(e).splitlines()[0]}")

## 14. 🆕 MCP Tool Recording

Model Context Protocol (MCP) tool calls can be recorded and replayed just like regular tool calls. TraceOps captures the full MCP event lifecycle: connect → call → result.

In [ ]:
from trace_ops.mcp.events import MCPEvent, MCPEventType
from trace_ops.mcp.diff import diff_mcp_traces

# Build a trace with MCP events
mcp_trace = Trace(metadata=TraceMetadata(description="MCP file-system agent"))

mcp_trace.add_event(TraceEvent(
    event_type=EventType.MCP_SERVER_CONNECT,
    tool_name="filesystem-server",
    tool_input={"transport": "stdio", "command": "mcp-server-filesystem"},
))
mcp_trace.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_CALL,
    tool_name="read_file",
    tool_input={"path": "/data/report.txt"},
    model="gpt-4o",
))
mcp_trace.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_RESULT,
    tool_name="read_file",
    tool_output={"content": "Q1 revenue: $1.2M\nQ2 revenue: $1.5M"},
    duration_ms=12.4,
))
mcp_trace.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_CALL,
    tool_name="write_file",
    tool_input={"path": "/data/summary.txt", "content": "Total H1 revenue: $2.7M"},
))
mcp_trace.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_RESULT,
    tool_name="write_file",
    tool_output={"success": True},
    duration_ms=8.1,
))
mcp_trace.finalize()

print(f"MCP trace events : {len(mcp_trace.events)}")
mcp_events = [e for e in mcp_trace.events if "mcp" in e.event_type.value]
print(f"MCP events       : {len(mcp_events)}")
for e in mcp_trace.events:
    icon = {"mcp_server_connect": "🔌", "mcp_tool_call": "📡", "mcp_tool_result": "📬"}.get(e.event_type.value, "•")
    print(f"  {icon}  {e.event_type.value:25s}  tool={e.tool_name}")

# --- MCP-aware diff ---
mcp_trace_v2 = Trace(metadata=TraceMetadata(description="MCP agent v2 — switched to list_directory"))
mcp_trace_v2.add_event(TraceEvent(
    event_type=EventType.MCP_SERVER_CONNECT,
    tool_name="filesystem-server",
    tool_input={"transport": "stdio", "command": "mcp-server-filesystem"},
))
mcp_trace_v2.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_CALL,
    tool_name="list_directory",   # different tool!
    tool_input={"path": "/data/"},
))
mcp_trace_v2.add_event(TraceEvent(
    event_type=EventType.MCP_TOOL_RESULT,
    tool_name="list_directory",
    tool_output={"entries": ["report.txt", "summary.txt"]},
    duration_ms=5.2,
))
mcp_trace_v2.finalize()

mcp_diff = diff_mcp_traces(mcp_trace, mcp_trace_v2)
print(f"\n=== MCP Diff ===")
print(f"tool_sequence_changed : {mcp_diff.tool_sequence_changed}")
print(f"added_tools           : {mcp_diff.added_tools}")
print(f"removed_tools         : {mcp_diff.removed_tools}")
print(f"summary: {mcp_diff.summary()}")

## 15. 🆕 Fine-Tune Dataset Export

Export any cassette as an OpenAI or Anthropic fine-tuning JSONL file. Each LLM request/response pair becomes one training example.

In [ ]:
from trace_ops.export.finetune import export_finetune_jsonl, ExportFormat

export_dir = CASSETTE_DIR / "finetune"
export_dir.mkdir(exist_ok=True)

# Export the math agent trace as OpenAI fine-tune JSONL
openai_path = export_dir / "math_agent_openai.jsonl"
export_finetune_jsonl(trace, openai_path, fmt=ExportFormat.OPENAI)

# Export as Anthropic fine-tune JSONL
anthropic_path = export_dir / "math_agent_anthropic.jsonl"
export_finetune_jsonl(trace, anthropic_path, fmt=ExportFormat.ANTHROPIC)

# --- Inspect the outputs ---
for path in [openai_path, anthropic_path]:
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    print(f"\n{'='*60}")
    print(f"📄 {path.name}  ({len(lines)} training examples)")
    print(f"{'='*60}")
    for line in lines[:2]:   # show first 2 examples
        record = json.loads(line)
        print(json.dumps(record, indent=2)[:400])
        if len(json.dumps(record)) > 400:
            print("  ...")
    print(f"\n✅ Saved: {path}  ({path.stat().st_size} bytes)")

## 16. 🆕 Behavioral Pattern Analysis *(agent-pr-replay inspired)*

TraceOps v0.6.0 adds three analysis classes inspired by [`sshh12/agent-pr-replay`](https://github.com/sshh12/agent-pr-replay):

| Class | Inspired by | What it does |
|---|---|---|
| `PatternDetector` | `stats.py` | Tool n-gram heatmaps, model usage, error rates across cassette dirs |
| `GapAnalyzer` | `diff_comparison.py` | Compare agent traces vs golden baselines; surface critical/warning gaps |
| `SkillsGenerator` | `analyzer.generate_report` | Auto-generate AGENTS.md / CLAUDE.md guidance from gaps |
| `PRFetcher` | `pr_finder.py` + `repo.py` | Fetch merged GitHub PRs as golden baselines via the REST API |

In [ ]:
"""§16a — PatternDetector: find recurring tool sequences across many cassettes."""

from pathlib import Path
import tempfile

from trace_ops._types import EventType, Trace, TraceEvent, TraceMetadata
from trace_ops.cassette import save_cassette
from trace_ops.analysis import PatternDetector


def _quick_trace(tools, model="gpt-4o", tokens=120):
    """Helper — build a minimal Trace from a list of tool names."""
    t = Trace(metadata=TraceMetadata(description="demo"))
    t.add_event(TraceEvent(event_type=EventType.LLM_REQUEST, model=model,
                           messages=[{"role": "user", "content": "do task"}]))
    t.add_event(TraceEvent(event_type=EventType.LLM_RESPONSE, model=model,
                           input_tokens=tokens // 2, output_tokens=tokens // 2,
                           cost_usd=0.001))
    for tool in tools:
        t.add_event(TraceEvent(event_type=EventType.TOOL_CALL, tool_name=tool))
        t.add_event(TraceEvent(event_type=EventType.TOOL_RESULT, tool_name=tool,
                               tool_output="ok"))
    t.finalize()
    return t


# ── Build a small demo cassette directory ─────────────────────────────────────
with tempfile.TemporaryDirectory() as tmpdir:
    cassette_dir = Path(tmpdir)

    # Simulate 6 agent runs — most follow search → read → write
    demo_traces = [
        _quick_trace(["search", "read", "write"]),
        _quick_trace(["search", "read", "write"]),
        _quick_trace(["search", "read", "write", "diff"]),
        _quick_trace(["search", "read"]),
        _quick_trace(["list_dir", "read", "write"]),
        _quick_trace(["search", "read", "write"]),
    ]
    for i, trace in enumerate(demo_traces):
        save_cassette(trace, cassette_dir / f"run_{i:02d}.yaml")

    detector = PatternDetector(window_size=3, top_n=5)
    report = detector.analyze_dir(cassette_dir)

print(report.summary())
print()
print("Top tool sequences:")
for seq in report.top_tool_sequences:
    print(f"  {' → '.join(seq.sequence):<30} ×{seq.count}")
print()
print("Tool frequency (all tools):", dict(list(report.tool_frequency.items())[:6]))

In [ ]:
"""§16b — GapAnalyzer: compare agent traces to a golden baseline."""

from trace_ops.analysis import GapAnalyzer

# Golden: efficient agent — 100 tokens, always uses "search" + "write"
golden = [
    ("golden_01.yaml", _quick_trace(["search", "write"], tokens=100)),
    ("golden_02.yaml", _quick_trace(["search", "write"], tokens=120)),
    ("golden_03.yaml", _quick_trace(["search", "write"], tokens=110)),
]

# Agent under test: expensive, forgets "write", occasionally errors
def _error_trace():
    t = _quick_trace(["search"], tokens=600)  # 5× more tokens, missing "write"
    t.add_event(TraceEvent(event_type=EventType.ERROR,
                           error_type="TimeoutError", error_message="LLM timed out"))
    t.finalize()
    return t

agent = [
    ("agent_01.yaml", _quick_trace(["search"], tokens=550)),   # missing "write"
    ("agent_02.yaml", _quick_trace(["search"], tokens=620)),   # missing "write"
    ("agent_03.yaml", _error_trace()),                         # missing "write" + error
]

analyzer = GapAnalyzer()
gap_report = analyzer.compare(golden, agent)

print(gap_report.summary())
print()
for gap in gap_report.gaps:
    icon = {"critical": "🔴", "warning": "🟡", "info": "🔵"}.get(gap.severity, "•")
    print(f"{icon} [{gap.severity.upper()}] {gap.description}")
    if gap.golden_value is not None:
        print(f"   Golden: {gap.golden_value}  |  Agent: {gap.agent_value}")

In [ ]:
"""§16c — SkillsGenerator: auto-produce AGENTS.md guidance from gap report."""

from trace_ops.analysis import SkillsGenerator

gen = SkillsGenerator()

# From gap report → AGENTS.md style guidance
agents_md = gen.from_gap_report(
    gap_report,
    title="AGENTS.md — Efficiency Guidance for Search-Write Agent",
)
print(agents_md)

# From pattern report → pattern summary doc
print("\n" + "─" * 60 + "\n")

with tempfile.TemporaryDirectory() as tmpdir2:
    cassette_dir2 = Path(tmpdir2)
    for i, trace in enumerate(demo_traces):          # reuse traces from §16a
        save_cassette(trace, cassette_dir2 / f"run_{i:02d}.yaml")
    pattern_rpt = PatternDetector(window_size=2).analyze_dir(cassette_dir2)

pattern_md = gen.from_pattern_report(pattern_rpt, title="Pattern Summary — Demo Traces")
print(pattern_md)

In [ ]:
"""§16d — PRFetcher: fetch a merged GitHub PR as a golden baseline (mocked).

In production use `PRFetcher(token=os.environ["GITHUB_TOKEN"]).fetch(pr_url)`.
Here we use unittest.mock to avoid real HTTP calls so the demo is always runnable.
"""

import json
from unittest.mock import MagicMock, patch

from trace_ops.github import PRFetcher, PRDiff, PRFile


def _make_mock_http_response(data):
    resp = MagicMock()
    resp.read.return_value = json.dumps(data).encode()
    resp.__enter__ = lambda s: s
    resp.__exit__ = MagicMock(return_value=False)
    return resp


pr_api_payload = {
    "number": 1337,
    "title": "Fix: agent skips write step when search returns empty results",
    "body": (
        "Closes #1336.\n\n"
        "The agent was silently skipping the `write` step when `search` "
        "returned zero chunks. This PR adds an explicit guard and a fallback."
    ),
    "merged_at": "2024-06-01T10:00:00Z",
    "html_url": "https://github.com/acme/ai-agent/pull/1337",
    "user": {"login": "alice"},
}

files_payload = [
    {
        "filename": "agent/pipeline.py",
        "additions": 18,
        "deletions": 4,
        "status": "modified",
        "patch": (
            "@@ -42,7 +42,21 @@ def run_pipeline(query):\n"
            "-    results = search(query)\n"
            "+    results = search(query)\n"
            "+    if not results:\n"
            "+        logger.warning('search returned 0 chunks; using fallback')\n"
            "+        results = fallback_search(query)\n"
        ),
    },
    {
        "filename": "tests/test_pipeline.py",
        "additions": 12,
        "deletions": 0,
        "status": "added",
        "patch": "@@ -0,0 +1,12 @@\n+def test_empty_search_uses_fallback(): ...",
    },
]

with patch("trace_ops.github.pr_fetcher.urllib.request.urlopen") as mock_open:
    mock_open.side_effect = [
        _make_mock_http_response(pr_api_payload),
        _make_mock_http_response(files_payload),
    ]
    fetcher = PRFetcher()  # no real token needed — mocked
    pr_diff = fetcher.fetch("https://github.com/acme/ai-agent/pull/1337")

print("=== PR Metadata ===")
print(f"PR #{pr_diff.pr_number}: {pr_diff.title}")
print(f"Author: {pr_diff.author}  |  Merged: {pr_diff.merged_at}")
print(f"Changes: +{pr_diff.total_additions} -{pr_diff.total_deletions} lines "
      f"across {len(pr_diff.files)} file(s)")

print("\n=== Reverse-Engineered Task Prompt ===")
print(pr_diff.extract_task_prompt())

print("\n=== Unified Diff (excerpt) ===")
print(pr_diff.diff_text[:400])

## 17. 🆕 LLM-as-Judge Evaluation

TraceOps v0.6.0 adds a general-purpose **LLM-as-judge** evaluation framework.  Unlike RAG scorers (which only work on retrieval traces), this evaluates **any** agent trace against quality criteria like *correctness*, *helpfulness*, *safety*, and *tool_efficiency*.

The judge sends the trace transcript to an LLM, which returns structured scores on a 1–5 scale (normalised to 0–1).

**Built-in criteria:** `correctness`, `helpfulness`, `conciseness`, `safety`, `tool_efficiency`, `goal_completion`, `faithfulness`, `tone`

> No real API key is needed below — we inject a mock `_llm_caller` that simulates the judge response.

In [ ]:
"""§17a — LLM-as-Judge: evaluate a trace against quality criteria."""

import json
from trace_ops._types import EventType, Trace, TraceEvent, TraceMetadata
from trace_ops.eval import (
    LLMJudge, EvalCriteria, TraceEvaluation, CriterionScore,
    BUILTIN_CRITERIA, build_trace_summary,
    assert_eval_score, assert_passes_criteria, EvalAssertionError,
)

# ── Show all built-in criteria ──────────────────────────────────────────
print("Built-in evaluation criteria:")
print("─" * 60)
for name, crit in BUILTIN_CRITERIA.items():
    print(f"  {name:<20s}  {crit.description[:70]}...")
print(f"\nTotal: {len(BUILTIN_CRITERIA)} criteria available out of the box")

# ── Build a demo trace to evaluate ──────────────────────────────────────
demo_trace = Trace(metadata=TraceMetadata(description="Customer support agent"))

demo_trace.add_event(TraceEvent(
    event_type=EventType.LLM_REQUEST,
    provider="openai", model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful customer support agent."},
        {"role": "user", "content": "I can't log in to my account. My email is alice@example.com."},
    ],
))
demo_trace.add_event(TraceEvent(
    event_type=EventType.TOOL_CALL,
    tool_name="lookup_user",
    tool_input={"email": "alice@example.com"},
))
demo_trace.add_event(TraceEvent(
    event_type=EventType.TOOL_RESULT,
    tool_name="lookup_user",
    tool_output={"user_id": "u-1234", "status": "locked", "last_login": "2024-01-15"},
    duration_ms=45.0,
))
demo_trace.add_event(TraceEvent(
    event_type=EventType.LLM_RESPONSE,
    provider="openai", model="gpt-4o",
    response={"choices": [{"message": {"content":
        "I found your account (alice@example.com). It's currently locked — likely due to "
        "too many failed login attempts. I've sent a password reset link to your email. "
        "Please check your inbox and follow the instructions to regain access."
    }}]},
    input_tokens=120, output_tokens=55, cost_usd=0.0008, duration_ms=620.0,
))
demo_trace.finalize()

# ── Preview what the judge sees ─────────────────────────────────────────
print("\n\n=== Trace Transcript (what the judge reads) ===")
print("─" * 60)
transcript = build_trace_summary(demo_trace)
print(transcript)

In [ ]:
"""§17b — Run the LLM judge with a mock caller (no real API key needed)."""

# ── Mock LLM caller — simulates the judge's structured JSON response ────
def mock_judge_caller(system: str, user: str, model: str) -> tuple[str, int]:
    """Simulate an LLM judge response for demo purposes."""
    evaluations = [
        {"criterion": "correctness",     "score": 5, "reasoning": "Correctly identified the locked account and sent a reset link."},
        {"criterion": "helpfulness",     "score": 5, "reasoning": "Provided clear, actionable steps for the user to regain access."},
        {"criterion": "tool_efficiency", "score": 4, "reasoning": "Used lookup_user appropriately; only one tool call needed."},
        {"criterion": "safety",          "score": 5, "reasoning": "Did not expose sensitive data; handled PII appropriately."},
        {"criterion": "tone",            "score": 4, "reasoning": "Professional and empathetic, appropriate for support context."},
    ]
    return json.dumps({"evaluations": evaluations}), 280  # (response_text, token_count)


# ── Create a judge with 5 criteria ──────────────────────────────────────
judge = LLMJudge(
    model="gpt-4o-mini",
    criteria=["correctness", "helpfulness", "tool_efficiency", "safety", "tone"],
    _llm_caller=mock_judge_caller,
)

result = judge.evaluate(demo_trace)

# ── Display results ─────────────────────────────────────────────────────
print("=== LLM-as-Judge Evaluation Results ===")
print("─" * 60)
print(result.summary())

print(f"\n{'─' * 60}")
print(f"Overall score : {result.overall_score:.2f}")
print(f"Judge model   : {result.judge_model}")
print(f"Judge tokens  : {result.judge_tokens}")
print(f"Judge cost    : ${result.judge_cost_usd:.6f}")

# ── Programmatic access ─────────────────────────────────────────────────
print(f"\n=== Programmatic Access ===")
for s in result.scores:
    status = "✅" if s.score >= 0.7 else "⚠️" if s.score >= 0.4 else "❌"
    print(f"  {status} {s.criterion:<20s}  score={s.score:.2f}  raw={s.raw_score}/5  — {s.reasoning}")

# ── Check individual scores ─────────────────────────────────────────────
print(f"\nresult.passes('correctness', min_score=0.8) → {result.passes('correctness', min_score=0.8)}")
print(f"result.passes('tone', min_score=0.9)         → {result.passes('tone', min_score=0.9)}")
print(f"result.score_for('safety').score              → {result.score_for('safety').score}")

# ── Serialise to JSON ───────────────────────────────────────────────────
print(f"\n=== JSON Output ===")
print(json.dumps(result.to_dict(), indent=2))

In [ ]:
"""§17c — Eval assertions: pytest-friendly score checks."""

# ── assert_eval_score — all criteria must pass ─────────────────────────
print("=== Eval Assertions ===\n")

try:
    result = assert_eval_score(
        demo_trace,
        criteria=["correctness", "helpfulness", "safety"],
        min_score=0.6,
        judge=judge,
    )
    print(f"✅ assert_eval_score(min_score=0.6) — passed!  overall={result.overall_score:.2f}")
except EvalAssertionError as e:
    print(f"❌ {e}")

# ── assert_passes_criteria — shorthand ─────────────────────────────────
try:
    result = assert_passes_criteria(
        demo_trace,
        ["correctness", "tone"],
        min_score=0.7,
        judge=judge,
    )
    print(f"✅ assert_passes_criteria(['correctness', 'tone'], min_score=0.7) — passed!")
except EvalAssertionError as e:
    print(f"❌ {e}")

# ── Deliberately trigger a failure with a very high bar ─────────────────
print("\n--- Triggering a failure deliberately ---")

# Mock a low-scoring judge
def harsh_judge(system: str, user: str, model: str) -> tuple[str, int]:
    return json.dumps({"evaluations": [
        {"criterion": "correctness", "score": 2, "reasoning": "Response lacked detail on root cause."},
        {"criterion": "helpfulness", "score": 1, "reasoning": "Did not offer alternative recovery methods."},
        {"criterion": "safety",      "score": 5, "reasoning": "No PII leaked."},
    ]}), 150

harsh = LLMJudge(criteria=["correctness", "helpfulness", "safety"], _llm_caller=harsh_judge)

try:
    assert_eval_score(demo_trace, min_score=0.7, judge=harsh)
except EvalAssertionError as e:
    print(f"\n💥 EvalAssertionError caught:")
    for line in str(e).splitlines():
        print(f"   {line}")

## 18. 🆕 Custom Evaluation Criteria

The 8 built-in criteria cover common use cases, but you can define **custom criteria** for domain-specific evaluation — brand voice, regulatory compliance, citation quality, etc.

Custom `EvalCriteria` objects can be mixed freely with built-in criterion names.

In [ ]:
"""§18 — Custom criteria: brand voice, compliance, citation quality."""

# ── Define custom criteria ──────────────────────────────────────────────
brand_voice = EvalCriteria(
    name="brand_voice",
    description=(
        "Does the response match our company's brand voice? "
        "We aim for a warm, professional tone that avoids jargon "
        "and addresses the user by implication, never by name."
    ),
)

regulatory_compliance = EvalCriteria(
    name="regulatory_compliance",
    description=(
        "Does the response comply with GDPR and data protection regulations? "
        "It must not log, repeat, or expose PII (emails, phone numbers, addresses) "
        "in the response text."
    ),
)

citation_quality = EvalCriteria(
    name="citation_quality",
    description=(
        "If the agent retrieved documents or context, does the response accurately "
        "cite or reference the source material without fabricating information?"
    ),
    scale_min=1,
    scale_max=10,  # Custom 1-10 scale
)

print("Custom criteria defined:")
for c in [brand_voice, regulatory_compliance, citation_quality]:
    print(f"\n  📐 {c.name}")
    print(f"     Scale: {c.scale_min}–{c.scale_max}")
    print(f"     {c.prompt_text()}")

# ── Mix custom + built-in criteria in one judge ─────────────────────────
def custom_mock_caller(system: str, user: str, model: str) -> tuple[str, int]:
    """Mock caller that scores both built-in and custom criteria."""
    evaluations = [
        {"criterion": "correctness",           "score": 5, "reasoning": "Accurate diagnosis of locked account."},
        {"criterion": "safety",                "score": 5, "reasoning": "No unsafe content generated."},
        {"criterion": "brand_voice",           "score": 4, "reasoning": "Warm and professional; minor jargon ('locked') could be softened."},
        {"criterion": "regulatory_compliance", "score": 5, "reasoning": "Email was not repeated in the response text."},
        {"criterion": "citation_quality",      "score": 7, "reasoning": "Referenced the lookup result correctly."},  # on 1-10 scale
    ]
    return json.dumps({"evaluations": evaluations}), 350

custom_judge = LLMJudge(
    model="gpt-4o",
    criteria=["correctness", "safety", brand_voice, regulatory_compliance, citation_quality],
    _llm_caller=custom_mock_caller,
)

custom_result = custom_judge.evaluate(demo_trace)

print("\n\n=== Custom Criteria Evaluation ===")
print("─" * 60)
print(custom_result.summary())

# ── Note the citation_quality normalisation (1-10 scale → 0-1) ─────────
cq = custom_result.score_for("citation_quality")
print(f"\ncitation_quality: raw={cq.raw_score}/10 → normalised={cq.score:.2f}")
print(f"  (scale 1-10: (7-1)/(10-1) = {(7-1)/(10-1):.4f})")

# ── Eval in pytest: one-liner assertion ─────────────────────────────────
print("\n=== Assertion with Custom Criteria ===")
try:
    assert_eval_score(demo_trace, criteria=["correctness", "brand_voice"], min_score=0.6, judge=custom_judge)
    print("✅ Custom criteria assertion passed!")
except EvalAssertionError as e:
    print(f"❌ {e}")

## 19. 🆕 Eval CLI & Pytest Marker

The eval feature is also available from the **command line** and as a **pytest marker**.

### CLI: `traceops eval`
```bash
# Evaluate a cassette with default criteria
traceops eval cassettes/test.yaml

# Specific criteria + minimum score gate (exits 1 on failure)
traceops eval cassettes/test.yaml -c correctness,safety -m gpt-4o --min-score 0.7

# Save JSON report
traceops eval cassettes/test.yaml -o eval_report.json
```

### Pytest marker: `@pytest.mark.eval`
```python
@pytest.mark.eval(criteria=["correctness", "safety"], min_score=0.8)
def test_my_agent(cassette):
    agent.run("What is 2+2?")
    # After the test runs, the eval marker automatically evaluates
    # the recorded trace and fails if any criterion scores below 0.8
```

In [ ]:
"""§19a — CLI: traceops eval --help."""

import subprocess, sys

def run_cli(*args):
    """Run a traceops CLI command and print the output."""
    cmd = [sys.executable, "-m", "trace_ops.cli"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    print(output or "(no output)")
    return result

print("=" * 60)
print("traceops eval --help")
print("=" * 60)
run_cli("eval", "--help")

In [ ]:
"""§19b — Eval comparison: side-by-side scoring of two agent versions."""

# Let's evaluate the math_v1 and math_v2 traces from section 7 side-by-side
# to show how eval can catch quality regressions between agent versions.

from trace_ops.cassette import load_cassette

trace_v1 = load_cassette(CASSETTE_DIR / "math_v1.yaml")
trace_v2 = load_cassette(CASSETTE_DIR / "math_v2.yaml")

# Mock judge that gives different scores based on trace content
def version_comparison_judge(system: str, user: str, model: str) -> tuple[str, int]:
    """Score differently based on trace content to simulate real evaluation."""
    # Detect which version by checking if transcript mentions gpt-4o-mini
    if "gpt-4o-mini" in user:
        # v2: cheaper model, fewer steps, but slightly less thorough
        evaluations = [
            {"criterion": "correctness",     "score": 5, "reasoning": "Still got the right answer (12)."},
            {"criterion": "helpfulness",     "score": 3, "reasoning": "Shorter response, missing explanation of the calculation."},
            {"criterion": "tool_efficiency", "score": 5, "reasoning": "Fewer LLM round-trips — more efficient."},
            {"criterion": "conciseness",     "score": 5, "reasoning": "No unnecessary verbosity."},
        ]
    else:
        # v1: more thorough but more expensive
        evaluations = [
            {"criterion": "correctness",     "score": 5, "reasoning": "Correctly computed sqrt(144) = 12."},
            {"criterion": "helpfulness",     "score": 5, "reasoning": "Clear explanation with step-by-step reasoning."},
            {"criterion": "tool_efficiency", "score": 3, "reasoning": "Two LLM round-trips when one would suffice."},
            {"criterion": "conciseness",     "score": 3, "reasoning": "Could be more concise."},
        ]
    return json.dumps({"evaluations": evaluations}), 200

compare_judge = LLMJudge(
    criteria=["correctness", "helpfulness", "tool_efficiency", "conciseness"],
    _llm_caller=version_comparison_judge,
)

result_v1 = compare_judge.evaluate(trace_v1)
result_v2 = compare_judge.evaluate(trace_v2)

# ── Side-by-side comparison ─────────────────────────────────────────────
print("=== Eval Comparison: v1 (gpt-4o) vs v2 (gpt-4o-mini) ===")
print("─" * 70)
print(f"{'Criterion':<22} {'v1 (gpt-4o)':>12} {'v2 (gpt-4o-mini)':>18}  {'Delta':>8}")
print("─" * 70)

for s1 in result_v1.scores:
    s2 = result_v2.score_for(s1.criterion)
    if s2:
        delta = s2.score - s1.score
        arrow = "📈" if delta > 0 else "📉" if delta < 0 else "━━"
        print(f"  {s1.criterion:<20s} {s1.score:>10.2f}   {s2.score:>16.2f}  {delta:>+7.2f} {arrow}")

print("─" * 70)
print(f"  {'OVERALL':<20s} {result_v1.overall_score:>10.2f}   {result_v2.overall_score:>16.2f}  "
      f"{result_v2.overall_score - result_v1.overall_score:>+7.2f}")

print(f"\n💡 v2 trades helpfulness for efficiency — a deliberate optimisation.")
print(f"   Use eval assertions to ensure the tradeoff stays acceptable:")
print(f"   assert_eval_score(trace_v2, criteria=['helpfulness'], min_score=0.4)")